# Task 1: OpenAlex Data Collection
**Goal:** Collect scientific publications for Semiconductors and Gene Engineering domains from OpenAlex API.

**Output:** Cleaned Parquet files in `data/processed/` + publication dynamics metrics.

| Step | Description |
|------|-------------|
| 0 | Setup & imports |
| 1 | Volume analysis (yearly counts) |
| 2 | Data collection (ETL) |
| 3 | Raw data quality check |
| 4 | Data cleaning |
| 5 | Metrics (CAGR, YoY, Acceleration) |
| 6 | PMI (Patent Momentum Indicator) |
| 7 | Visualization |
| 8 | Conclusions |

In [ ]:
# 0. Setup 
import sys
import json
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/semiconductor_project'
sys.path.append(PROJECT_PATH)

CONFIG_PATH = PROJECT_PATH + '/config/domains.json'
RAW_DIR     = PROJECT_PATH + '/data/raw'
PROC_DIR    = PROJECT_PATH + '/data/processed'

with open(CONFIG_PATH) as f:
    config = json.load(f)

# Domains for this run
# In Streamlit this will come from st.multiselect()
SELECTED_DOMAINS = ['semiconductors', 'gene_engineering']

print('Config loaded. Available domains:', list(config.keys()))

In [ ]:
# 1. Volume Analysis 
# Check how many papers exist per year before downloading
from etl.openalex_client import load_domain_config, get_yearly_counts, print_yearly_counts

for domain_key in SELECTED_DOMAINS:
    meta = config[domain_key]
    counts = get_yearly_counts(meta['openalex_topic_ids'])
    print_yearly_counts(meta['display_name'], counts)

In [ ]:
# 2. Data Collection (ETL) 
# Downloads papers via cursor-based pagination.
# Supports resume after interruption (cursor.json)
# and incremental reload when new Topic IDs are added (manifest.json)
from etl.openalex_client import collect_works

for domain_key in SELECTED_DOMAINS:
    collect_works(
        domain=domain_key,
        save_dir=RAW_DIR + '/' + domain_key
    )

In [ ]:
# 3. Raw Data Quality Check 
import glob

for domain_key in SELECTED_DOMAINS:
    files = glob.glob(RAW_DIR + '/' + domain_key + '/*.parquet')
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

    print('=' * 40)
    print('Domain:          ', config[domain_key]['display_name'])
    print('Total records:   ', len(df))
    print('Unique IDs:      ', df['id'].nunique())
    print('Duplicates:      ', len(df) - df['id'].nunique())
    print('Empty title:     ', df['title'].isna().sum())
    print('Empty abstract:  ', df['abstract'].eq('').sum())
    print('Empty topic_id:  ', df['topic_id'].eq('').sum())
    print('Years:           ', df['year'].min(), '–', df['year'].max())
    print()

In [ ]:
# 4. Data Cleaning 
# Removes: duplicates by id, records without title,
# records without abstract (required for NLP in Task 5)
# Raw data stays untouched in data/raw/
from etl.preprocessing import clean_domain_data

for domain_key in SELECTED_DOMAINS:
    clean_domain_data(
        domain=domain_key,
        raw_dir=RAW_DIR,
        processed_dir=PROC_DIR
    )

In [ ]:
# 5. Metrics 
# CAGR: compound annual growth rate
# YoY:  year-over-year growth
# Acceleration: change in growth rate between two windows
from etl.metrics import calc_domain_summary, print_domain_summary

summaries = []

for domain_key in SELECTED_DOMAINS:
    df = pd.read_parquet(PROC_DIR + '/' + domain_key + '_clean.parquet')
    df['publication_date'] = pd.to_datetime(df['publication_date'], errors='coerce')
    df['period'] = df['publication_date'].dt.to_period('M')

    monthly = df.groupby('period').size().reset_index(name='count')
    monthly['period_dt'] = monthly['period'].dt.to_timestamp()
    yearly  = df[df['year'] < 2025].groupby('year').size().reset_index(name='count')

    summary = calc_domain_summary(domain_key, config[domain_key]['display_name'], yearly, monthly)
    print_domain_summary(summary)
    summaries.append(summary)

In [ ]:
# 6. PMI (Patent Momentum Indicator) 
# PMI = Z(Activity) + Z(CAGR)
# With 2 domains shows relative positioning only.
from etl.metrics import calc_pmi, print_pmi_report

result = calc_pmi(summaries)
print_pmi_report(result)

In [ ]:
# 7. Visualization 
# Reusable function — same code will be used in Streamlit (Task 8)
from visualization.charts import plot_publications_dynamics

for domain_key in SELECTED_DOMAINS:
    df = pd.read_parquet(PROC_DIR + '/' + domain_key + '_clean.parquet')
    df['publication_date'] = pd.to_datetime(df['publication_date'], errors='coerce')
    df['period'] = df['publication_date'].dt.to_period('M')

    monthly = df.groupby('period').size().reset_index(name='count')
    monthly['period_dt'] = monthly['period'].dt.to_timestamp()
    yearly  = df[df['year'] < 2026].groupby('year').size().reset_index(name='count')

    fig = plot_publications_dynamics(
        domain_key=domain_key,
        label=config[domain_key]['display_name'],
        color=config[domain_key]['color'],
        monthly_df=monthly,
        yearly_df=yearly
    )
    fig.show()

## Conclusions

### Semiconductors
- **CAGR −3.1%** — steady structural decline since the 2011 peak (24,850 papers).
- Fundamental science has matured; the domain is shifting toward applied R&D and patenting.
- The 2023 rebound (+10% YoY) is explained by the global chip crisis and state investment programs (US CHIPS Act). By 2024 the decline resumed.
- **Acceleration −0.3%** — the rate of decline has stabilized; no free fall.

### Gene Engineering
- **CAGR −0.6%** — effectively stagnation, not decline.
- Two distinct peaks: **2016** (CRISPR boom, first clinical trials) and **2019** (base/prime editing wave).
- The 2017–2018 dip likely reflects activity shifting from open publications to patenting.
- Stabilization at 13–14k papers/year since 2022 indicates a mature steady state.

### Overall
Declining publication activity does **not** mean technological decline. This is the classic pattern of mature domains where the center of gravity shifts from fundamental science to commercialization and patenting.

**Key hypothesis for Tasks 4 & 7:** If patent activity is growing while publications decline → this is the main signal of technology transfer that the platform is built to detect.